<a href="https://colab.research.google.com/github/israfilisrafilov/Israfil-Israfilov-DataScience-GenAI-Submissions/blob/main/6_02_DNN_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [2]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [3]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [4]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [6]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [7]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [8]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [9]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 31470.6914
Epoch [10/100], Loss: 31901.6074
Epoch [10/100], Loss: 26025.1797
Epoch [10/100], Loss: 27777.2637
Epoch [10/100], Loss: 26358.2773
Epoch [10/100], Loss: 26763.9785
Epoch [10/100], Loss: 38119.0273
Epoch [10/100], Loss: 12135.5273
Epoch [20/100], Loss: 26789.5039
Epoch [20/100], Loss: 32723.4551
Epoch [20/100], Loss: 23041.7637
Epoch [20/100], Loss: 19745.7637
Epoch [20/100], Loss: 44964.4727
Epoch [20/100], Loss: 34126.4141
Epoch [20/100], Loss: 26030.7617
Epoch [20/100], Loss: 12138.1094
Epoch [30/100], Loss: 24809.2539
Epoch [30/100], Loss: 28928.1719
Epoch [30/100], Loss: 33293.1797
Epoch [30/100], Loss: 27629.0762
Epoch [30/100], Loss: 28499.4336
Epoch [30/100], Loss: 29285.75
Epoch [30/100], Loss: 33455.5234
Epoch [30/100], Loss: 11176.4453
Epoch [40/100], Loss: 28073.9453
Epoch [40/100], Loss: 29820.7539
Epoch [40/100], Loss: 27801.5938
Epoch [40/100], Loss: 28554.3984
Epoch [40/100], Loss: 26825.8262
Epoch [40/100], Loss: 28608.6074
Epoch [40/10

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [10]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 19344.1396484375


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [11]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,27.391453,219.0
1,25.324894,70.0
2,26.814762,202.0
3,33.322208,230.0
4,26.028755,111.0
...,...,...
84,23.385944,153.0
85,21.437572,98.0
86,19.088192,37.0
87,20.047747,63.0


In [1]:

import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())


Num GPUs Available:  1


In [12]:

# Upload an in-built Python dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# Import the data
data = load_diabetes()
data


{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

In [14]:

# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1)


In [15]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


In [16]:

from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)


In [17]:

import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()

        self.layers = nn.Sequential(
            nn.Linear(10, 5),
            nn.ReLU(),

            nn.Linear(5, 5),
            nn.ReLU(),

            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x)


In [18]:

import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)


In [19]:

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop
epochs = 100

for epoch in range(epochs):
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        model.to(device)
        model.train()
        optimiser.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimiser.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')


Epoch [10/100], Loss: 33988.8281
Epoch [10/100], Loss: 26892.9766
Epoch [10/100], Loss: 27269.7852
Epoch [10/100], Loss: 26701.8242
Epoch [10/100], Loss: 30748.8965
Epoch [10/100], Loss: 27730.7852
Epoch [10/100], Loss: 32045.2949
Epoch [10/100], Loss: 68876.5078
Epoch [20/100], Loss: 33870.4141
Epoch [20/100], Loss: 26089.6836
Epoch [20/100], Loss: 30346.5391
Epoch [20/100], Loss: 27794.1191
Epoch [20/100], Loss: 30353.3984
Epoch [20/100], Loss: 27587.3145
Epoch [20/100], Loss: 29860.4922
Epoch [20/100], Loss: 48232.5547
Epoch [30/100], Loss: 26755.8125
Epoch [30/100], Loss: 29983.0996
Epoch [30/100], Loss: 34749.3359
Epoch [30/100], Loss: 30578.3242
Epoch [30/100], Loss: 33843.043
Epoch [30/100], Loss: 24825.3203
Epoch [30/100], Loss: 26416.75
Epoch [30/100], Loss: 6712.5391
Epoch [40/100], Loss: 18291.957
Epoch [40/100], Loss: 33682.7383
Epoch [40/100], Loss: 26450.9141
Epoch [40/100], Loss: 37739.3125
Epoch [40/100], Loss: 27960.2988
Epoch [40/100], Loss: 30876.8652
Epoch [40/100],

In [20]:

# Evaluation
model.eval()
mse_values = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets)
        mse_values.append(mse.item())

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")


Average MSE on test set: 20958.408203125


In [21]:

# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({
    'Predicted': np.array(predictions).flatten(),
    'Actual': np.array(actuals).flatten()
})

results_df


,Predicted,Actual
0,20.216475,219.0
1,19.217909,70.0
2,19.846415,202.0
3,24.139799,230.0
4,19.419849,111.0
...,...,...
84,17.482897,153.0
85,16.450809,98.0
86,14.956613,37.0
87,15.588262,63.0


Side-by-side, they don't look great. Can you improve them?


EXERCISE #1

Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

In [22]:

# Exercise 1: Train for 1000 epochs

model = DiabetesModel().to(device)
optimiser = optim.Adam(model.parameters(), lr=0.001)

epochs = 1000

for epoch in range(epochs):
    model.train()

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimiser.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        optimiser.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [100/1000], Loss: 25964.8125
Epoch [200/1000], Loss: 11465.0293
Epoch [300/1000], Loss: 5464.9062
Epoch [400/1000], Loss: 4237.4258
Epoch [500/1000], Loss: 2835.8074
Epoch [600/1000], Loss: 2729.0171
Epoch [700/1000], Loss: 2748.3679
Epoch [800/1000], Loss: 559.7805
Epoch [900/1000], Loss: 1139.2095
Epoch [1000/1000], Loss: 1008.6223


In [23]:

# Evaluate model after 1000 epochs
model.eval()
mse_values = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)
        mse = criterion(outputs, targets)
        mse_values.append(mse.item())

avg_mse_1000 = np.mean(mse_values)

print(f"MSE after 100 epochs: 20958.41")
print(f"MSE after 1000 epochs: {avg_mse_1000:.2f}")


MSE after 100 epochs: 20958.41
MSE after 1000 epochs: 3002.10


EXERCISE #2 (optional)

Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

In [24]:

# Exercise 2: Change the model architecture

class DiabetesModel2(nn.Module):
    def __init__(self):
        super(DiabetesModel2, self).__init__()

        self.layers = nn.Sequential(
            nn.Linear(10, 20),
            nn.ReLU(),

            nn.Linear(20, 10),
            nn.ReLU(),

            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.layers(x)


In [25]:

# Initialize the new model
model2 = DiabetesModel2().to(device)

criterion2 = nn.MSELoss()
optimiser2 = optim.Adam(model2.parameters(), lr=0.001)

# Train the new model
epochs = 1000

for epoch in range(epochs):
    model2.train()

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimiser2.zero_grad()

        outputs = model2(inputs)
        loss = criterion2(outputs, targets)

        loss.backward()
        optimiser2.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [100/1000], Loss: 3901.2820
Epoch [200/1000], Loss: 3238.2156
Epoch [300/1000], Loss: 3238.9780
Epoch [400/1000], Loss: 5508.9761
Epoch [500/1000], Loss: 2970.6042
Epoch [600/1000], Loss: 1935.8665
Epoch [700/1000], Loss: 3627.2676
Epoch [800/1000], Loss: 280.0988
Epoch [900/1000], Loss: 2857.6377
Epoch [1000/1000], Loss: 1732.6340


In [26]:

# Evaluate the new model
model2.eval()
mse_values2 = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model2(inputs)
        mse = criterion2(outputs, targets)
        mse_values2.append(mse.item())

avg_mse_model2 = np.mean(mse_values2)

# Compare results
print("MODEL COMPARISON")
print("---------------------------")
print(f"Original Model (100 epochs): 20958.41")
print(f"Original Model (1000 epochs): {avg_mse_1000:.2f}")
print(f"New Model (1000 epochs): {avg_mse_model2:.2f}")

if avg_mse_model2 < avg_mse_1000:
    print("New architecture performed better!")
else:
    print("Original architecture performed better!")


MODEL COMPARISON
---------------------------
Original Model (100 epochs): 20958.41
Original Model (1000 epochs): 3002.10
New Model (1000 epochs): 2850.00
New architecture performed better!
